In [1]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="",
    port=5432
)

cur = conn.cursor()

# 转化漏斗分析：View -> Cart -> Purchase
query_funnel = """
WITH user_product_events AS (
    SELECT 
        user_id,
        product_id,
        STRING_AGG(event_type, ',' ORDER BY event_time) as event_sequence
    FROM makeup_consumer_events.dec
    GROUP BY user_id, product_id
)
SELECT 
    -- View -> Cart 转化率
    COUNT(*) as total_with_view,
    SUM(CASE WHEN event_sequence ILIKE '%cart%' THEN 1 ELSE 0 END) as total_with_cart,
    ROUND(100.0 * SUM(CASE WHEN event_sequence ILIKE '%cart%' THEN 1 ELSE 0 END) 
          / COUNT(*), 2) as view_to_cart_rate_pct,
    
    -- Cart -> Purchase 转化率
    SUM(CASE WHEN event_sequence ILIKE '%cart%purchase%' THEN 1 ELSE 0 END) as cart_to_purchase_count,
    ROUND(100.0 * SUM(CASE WHEN event_sequence ILIKE '%cart%purchase%' THEN 1 ELSE 0 END) 
          / NULLIF(SUM(CASE WHEN event_sequence ILIKE '%cart%' THEN 1 ELSE 0 END), 0), 2) as cart_to_purchase_rate_pct
FROM user_product_events
WHERE event_sequence ILIKE '%view%';
"""

cur.execute(query_funnel)
result = cur.fetchone()

columns = [desc[0] for desc in cur.description]
result_dict = dict(zip(columns, result))
result_df = pd.DataFrame([result_dict])
mapping = {'total_with_view': '总浏览数', 'total_with_cart': '加入购物车数', 'view_to_cart_rate_pct': '浏览转加入购物车率(%)',
           'cart_to_purchase_count': '加入购物车后购买数', 'cart_to_purchase_rate_pct': '加入购物车转购买率(%)'}
result_df = result_df.rename(columns=mapping)
desired_order = ['总浏览数', '加入购物车数',  '加入购物车后购买数', '浏览转加入购物车率(%)', '加入购物车转购买率(%)']
result_df = result_df[desired_order]
print(result_df)

cur.close()
conn.close()



      总浏览数  加入购物车数  加入购物车后购买数 浏览转加入购物车率(%) 加入购物车转购买率(%)
0  1270675  265959      72041        20.93        27.09


In [2]:
# !pip install plotly  # 如果报错说找不到 plotly 模块，把这行开头的 # 去掉执行一次即可
import plotly.graph_objects as go
import os

outpath = r"C:\Users\Administrator\Desktop\data_learn\eCommerce_Events_History\reports"

# 提取步骤和对应的绝对数值
stages = ['总浏览', '加入购物车', '购买']
values = [
    result_df['总浏览数'][0], 
    result_df['加入购物车数'][0], 
    result_df['加入购物车后购买数'][0]
]

# 自定义每层显示的文本，把绝对值和你算出的转化率拼在一起
custom_text = [
    f"浏览人数: {values[0]}",
    f"加购人数: {values[1]} <br>转化率: {result_df['浏览转加入购物车率(%)'][0]}%",
    f"购买人数: {values[2]} <br>转化率: {result_df['加入购物车转购买率(%)'][0]}%"
]

# 生成漏斗图
fig = go.Figure(go.Funnel(
    y = stages,
    x = values,
    textinfo = "text",
    text = custom_text
))

fig.update_layout(title_text="用户行为漏斗图 (View -> Cart -> Purchase)")
fig.show()

# 1. 定义文件名并拼接完整路径
file_name = "user_funnel.html"
full_path = os.path.join(outpath, file_name)

# 2. 导出文件
fig.write_html(full_path)

# 1. 修改文件名为 png 后缀
file_name = "user_funnel.png"
full_path = os.path.join(outpath, file_name)

# 2. 导出为静态图片
# 注意：如果此处报错，通常是缺少 kaleido 库
fig.write_image(full_path)

print(f"图片已保存至: {full_path}")

图片已保存至: C:\Users\Administrator\Desktop\data_learn\eCommerce_Events_History\reports\user_funnel.png
